# Data Preparation for Machine Learning — T01–T06
Trusted reproducible notebook, not hosted learner execution. Predict before running each example; guided changes belong in a private copy and never award platform mastery. All data are original fictional teaching records. No credentials, uploads, paid compute, Drive or remote datasets. Genuine Colab remains UNVERIFIED; do not install floating alternatives to make preflight pass.

In [ ]:
import os, sys, time, json, hashlib, resource
os.environ.update({"OMP_NUM_THREADS":"1","OPENBLAS_NUM_THREADS":"1","MKL_NUM_THREADS":"1"})
started = time.monotonic()
import numpy as np
import pandas as pd
import matplotlib
import sklearn
assert sys.version_info[:3] == (3,13,16)
assert (np.__version__,pd.__version__,matplotlib.__version__,sklearn.__version__) == ("2.5.4","3.0.6","3.11.2","1.9.1")
seed = 42
results = {}
def dtype_name(dtype):
    return f"string[{dtype.storage}]" if isinstance(dtype, pd.StringDtype) else str(dtype)

def normalize_table(value):
    if isinstance(value, pd.DataFrame):
        assert value.shape[0] <= 32 and value.shape[1] <= 8
        return {"kind": "frame", "shape": list(value.shape), "index": normalize_table(value.index.tolist()), "index_names": normalize_table(value.index.names), "columns": normalize_table(value.columns.tolist()), "dtypes": [dtype_name(dtype) for dtype in value.dtypes], "rows": normalize_table(value.values.tolist())}
    if isinstance(value, pd.Series):
        assert len(value) <= 32
        return {"kind": "series", "name": normalize_table(value.name), "index": normalize_table(value.index.tolist()), "index_names": normalize_table(value.index.names), "dtype": dtype_name(value.dtype), "values": normalize_table(value.tolist())}
    if value is pd.NA or value is pd.NaT:
        return {"tag": "missing"}
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return normalize_table(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return {"tag": "missing"} if np.isnan(value) else {"tag": "nonfinite", "value": "+inf" if value > 0 else "-inf"}
    if isinstance(value, dict):
        return {key: normalize_table(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [normalize_table(item) for item in value]
    if isinstance(value, str):
        assert len(value) <= 120
    if isinstance(value, (float, int)) and not isinstance(value, bool):
        assert abs(value) <= 1e12
    return value


def normalize(value):
    if isinstance(value, np.ndarray): return normalize(value.tolist())
    if isinstance(value, dict): return {key: normalize(item) for key,item in value.items()}
    if isinstance(value, (list,tuple)): return [normalize(item) for item in value]
    return normalize_table(value)


## T01 — Split observations before learning from them
Prerequisite recall: P03, D06; Statistics sampling/leakage.

Evaluation must respect the future-use question and unit of independence. Independent evaluation and sample-selection assumptions.

Terminology: feature, target, train, validation, test, group, time, random_state.

A fixed housing split uses 12 training IDs and 4 final-test IDs. Repeated observations from one customer must not be scattered across both sides of a customer-level generalization test.

Change the conditions: Compare fixed row split with group/time overlap and final-test-guided selection.

### T01.O1 — critical decision AND reason: Choose a split unit consistent with the intended generalization question.
Predict the output and which rows may fit before running. The unit of independence is the customer for an unseen-customer claim. Repeated rows from the same customer cannot straddle this boundary. A row-level split or a seed alone cannot establish independence.

Misconception: Any random row split is valid for new customers, even when their other records were used to fit.

Success: Reject an entity/time overlap and explain the unit of independence.

Remediation: Group row IDs by entity/time before splitting.

Guided change: Compare fixed row split with group/time overlap and final-test-guided selection. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train_entities = ['C1', 'C1', 'C2', 'C3']
evaluation_entities = ['C1', 'C4']
overlap = sorted(set(train_entities) & set(evaluation_entities))
result = {'question': 'generalize to unseen customers', 'entity_overlap': overlap, 'valid_customer_boundary': not overlap, 'repair': 'split by customer before learning'}
expected = json.loads("{\"question\":\"generalize to unseen customers\",\"entity_overlap\":[\"C1\"],\"valid_customer_boundary\":false,\"repair\":\"split by customer before learning\"}")
assert normalize(result) == expected
results["T01.O1"] = normalize(result)
print(json.dumps({"objective":"T01.O1","output":normalize(result)}, sort_keys=True))

### T01.O2 — critical decision AND reason: Keep the final test set untouched during preprocessing/model choice.
Predict the output and which rows may fit before running. Use development-training rows to fit and validation rows to compare declared policies. The final test is untouched during choice; even unsupervised statistics learned from it leak evaluation information.

Misconception: It is safe to choose an imputation policy using the final test score because no test target is passed to fit.

Success: Reject a test-guided choice and justify a validation alternative.

Remediation: Draw train/validation/test information permissions.

Guided change: Compare fixed row split with group/time overlap and final-test-guided selection. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train_ids = ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']
validation_ids = train_ids[3::4]
fold_train_ids = [identity for identity in train_ids if identity not in validation_ids]
result = {'fit_ids': fold_train_ids, 'policy_choice_ids': validation_ids, 'final_test_ids': ['H13', 'H14', 'H15', 'H16'], 'test_may_select_policy': False}
expected = json.loads("{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H05\",\"H06\",\"H07\",\"H09\",\"H10\",\"H11\"],\"policy_choice_ids\":[\"H04\",\"H08\",\"H12\"],\"final_test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"test_may_select_policy\":false}")
assert normalize(result) == expected
results["T01.O2"] = normalize(result)
print(json.dumps({"objective":"T01.O2","output":normalize(result)}, sort_keys=True))

### T01.O3: Predict a deterministic fixed-ID or seeded split and its limitations.
Predict the output and which rows may fit before running. Reproducibility records source order, IDs, split configuration and seed together. Fixed membership does not prove representativeness or remove group/time dependence.

Misconception: A random_state guarantees representative observations and is sufficient even if source ordering changes.

Success: Choose reproducible memberships without claiming representativeness.

Remediation: Compare seed, ordering and split configuration together.

Guided change: Compare fixed row split with group/time overlap and final-test-guided selection. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
ids = ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']
train, validation = train_test_split(ids, test_size=4, random_state=42, shuffle=True)
result = {'train_ids': train, 'validation_ids': validation, 'seed': 42, 'representative_guarantee': False, 'source_order_revision': 3}
expected = json.loads("{\"train_ids\":[\"H06\",\"H03\",\"H02\",\"H12\",\"H05\",\"H08\",\"H04\",\"H07\"],\"validation_ids\":[\"H11\",\"H10\",\"H01\",\"H09\"],\"seed\":42,\"representative_guarantee\":false,\"source_order_revision\":3}")
assert normalize(result) == expected
results["T01.O3"] = normalize(result)
print(json.dumps({"objective":"T01.O3","output":normalize(result)}, sort_keys=True))

### T01.O4: Identify feature availability and target timing before modeling.
Predict the output and which rows may fit before running. Feature availability is judged at prediction time, not by correlation. The target and after-sale information cannot be supplied for a genuinely new house.

Misconception: Any highly predictive future field is legitimate because it improves apparent validation accuracy.

Success: Exclude unavailable future information from a new record schema.

Remediation: Build a prediction-time feature timeline.

Guided change: Compare fixed row split with group/time overlap and final-test-guided selection. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
available = ['area', 'age']
unavailable = ['price', 'future_price', 'after_sale_3']
result = {'prediction_time_features': available, 'excluded': unavailable, 'reason': 'target or observed only after prediction'}
expected = json.loads("{\"prediction_time_features\":[\"area\",\"age\"],\"excluded\":[\"price\",\"future_price\",\"after_sale_3\"],\"reason\":\"target or observed only after prediction\"}")
assert normalize(result) == expected
results["T01.O4"] = normalize(result)
print(json.dumps({"objective":"T01.O4","output":normalize(result)}, sort_keys=True))

## T02 — Fit missing-data rules on training data
Prerequisite recall: T01, D03.

An imputer learns a rule from training data, then applies that unchanged rule elsewhere. Training-only medians and conditional missingness assumptions.

Terminology: imputer, fit, transform, training statistic, missing indicator.

Train ages [10,missing,30] fit median 20. Test ages [1000,missing] transform to [1000,20], not [1000,1000].

Change the conditions: Change only test values and verify the training median is unchanged.

### T02.O1 — critical decision AND reason: Fit imputation statistics using training observations only.
Predict the output and which rows may fit before running. Fit the median on training rows only: (10+30)/2=20. Transform test rows with20. The whole-data comparator is explicitly invalid even if a particular sample accidentally produces the same statistic.

Misconception: Unsupervised imputation can inspect every test value before splitting because it does not read targets.

Success: Repair a whole-dataset median fit and explain its leakage.

Remediation: Separate fit rows from transform-only rows.

Guided change: Change only test values and verify the training median is unchanged. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = np.array([[10.], [np.nan], [30.]])
test = np.array([[1000.], [np.nan]])
imputer = SimpleImputer(strategy='median').fit(train)
leaked = SimpleImputer(strategy='median').fit(np.vstack([train, test]))
result = {'fit_ids': ['H01','H02','H03'], 'training_median': imputer.statistics_, 'test_output': imputer.transform(test), 'invalid_whole_data_median': leaked.statistics_, 'whole_data_fit_allowed': False}
expected = json.loads("{\"fit_ids\":[\"H01\",\"H02\",\"H03\"],\"training_median\":[20],\"test_output\":[[1000],[20]],\"invalid_whole_data_median\":[30],\"whole_data_fit_allowed\":false}")
assert normalize(result) == expected
results["T02.O1"] = normalize(result)
print(json.dumps({"objective":"T02.O1","output":normalize(result)}, sort_keys=True))

### T02.O2: Transform new/missing records with the saved training rule.
Predict the output and which rows may fit before running. The saved20 is applied unchanged to a new missing cell. A changed test extreme can change its own transformed value but not the fitted training statistic.

Misconception: Each incoming test batch should fit its own median so the transformed batch looks typical.

Success: Predict values under a changed test batch.

Remediation: Apply the unchanged fitted statistic explicitly.

Guided change: Change only test values and verify the training median is unchanged. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = np.array([[10.], [np.nan], [30.]])
imputer = SimpleImputer(strategy='median').fit(train)
result = {'saved_statistic': imputer.statistics_, 'changed_test': imputer.transform([[1000.], [np.nan]]), 'refitted': False}
expected = json.loads("{\"saved_statistic\":[20],\"changed_test\":[[1000],[20]],\"refitted\":false}")
assert normalize(result) == expected
results["T02.O2"] = normalize(result)
print(json.dumps({"objective":"T02.O2","output":normalize(result)}, sort_keys=True))

### T02.O3: Choose and justify a missingness policy/indicator for the stated data.
Predict the output and which rows may fit before running. Keep observed zero distinct from absence. A missing indicator records information about absence, but does not prove why it happened; median filling is an assumption that may bias results.

Misconception: A filled value becomes an observed fact, and a missing indicator proves the cause of missingness.

Success: Select an appropriate bounded policy and acknowledge its assumption.

Remediation: Compare missing-pattern evidence and possible reasons.

Guided change: Change only test values and verify the training median is unchanged. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = np.array([[0.], [np.nan], [20.]])
imputer = SimpleImputer(strategy='median', add_indicator=True).fit(train)
result = {'training_statistic': imputer.statistics_, 'columns': ['age', 'age_was_missing'], 'prepared': imputer.transform(train), 'assumption': 'median is a declared approximation, not an observed fact'}
expected = json.loads("{\"training_statistic\":[10],\"columns\":[\"age\",\"age_was_missing\"],\"prepared\":[[0,0],[10,1],[20,0]],\"assumption\":\"median is a declared approximation, not an observed fact\"}")
assert normalize(result) == expected
results["T02.O3"] = normalize(result)
print(json.dumps({"objective":"T02.O3","output":normalize(result)}, sort_keys=True))

### T02.O4: Handle all-missing or unexpected columns explicitly.
Predict the output and which rows may fit before running. Inspect the pinned imputer's all-missing-column output instead of silently assuming a constant shape. A schema mismatch requires an explicit error or reviewed policy, not guessed column deletion/reordering.

Misconception: Every fitted imputer always preserves every column, so all-missing or reordered columns need no inspection.

Success: Choose a schema/error policy instead of silently deleting changing columns.

Remediation: Inspect feature names and fitted output schema.

Guided change: Change only test values and verify the training median is unchanged. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = pd.DataFrame({'age': [np.nan, np.nan], 'area': [1., 5.]})
imputer = SimpleImputer(strategy='median').fit(train)
result = {'fitted_features': imputer.feature_names_in_, 'statistics': imputer.statistics_, 'output_names': imputer.get_feature_names_out(), 'output': imputer.transform(train), 'schema_policy': 'report empty-column behavior; reject unexpected or reordered new columns'}
expected = json.loads("{\"fitted_features\":[\"age\",\"area\"],\"statistics\":[{\"tag\":\"missing\"},3],\"output_names\":[\"area\"],\"output\":[[1],[5]],\"schema_policy\":\"report empty-column behavior; reject unexpected or reordered new columns\"}")
assert normalize(result) == expected
results["T02.O4"] = normalize(result)
print(json.dumps({"objective":"T02.O4","output":normalize(result)}, sort_keys=True))

## T03 — Categorical encoding preserves a feature contract
Prerequisite recall: T01–T02, N03, D01.

Encoding creates a fitted coordinate vocabulary with an explicit unknown policy. Coordinate representation; arbitrary category codes need not express distance.

Terminology: category, ordinal, one-hot encoding, unknown category, fitted vocabulary.

Train plans [basic,pro] with fixed category order [basic,pro] encode basic=[1,0]. With handle_unknown='ignore', unseen team encodes [0,0], an explicit ambiguity rather than a new learned category.

Change the conditions: Compare a known category with an unseen category and arbitrary integer coding.

### T03.O1 — critical decision AND reason: Fit vocabulary on training data and apply an explicit unknown-category policy.
Predict the output and which rows may fit before running. The training vocabulary stays basic/pro. With ignore and no drop, an unseen category becomes[0,0]; it is not newly learned. Record the ambiguity or choose the declared error policy, never refit on test.

Misconception: Refit the encoder on evaluation categories so each new batch gets a complete and trustworthy feature vocabulary.

Success: Repair test refitting and explain the chosen unknown output.

Remediation: Inspect the saved category list and output column names.

Guided change: Compare a known category with an unseen category and arbitrary integer coding. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
encoder = OneHotEncoder(categories=[['basic','pro']], handle_unknown='ignore', drop=None, sparse_output=False).fit([['basic'],['pro']])
result = {'fit_ids': ['H01','H02'], 'categories': encoder.categories_, 'columns': encoder.get_feature_names_out(['plan']), 'unknown': 'team', 'encoded': encoder.transform([['team']]), 'test_refit_allowed': False}
expected = json.loads("{\"fit_ids\":[\"H01\",\"H02\"],\"categories\":[[\"basic\",\"pro\"]],\"columns\":[\"plan_basic\",\"plan_pro\"],\"unknown\":\"team\",\"encoded\":[[0,0]],\"test_refit_allowed\":false}")
assert normalize(result) == expected
results["T03.O1"] = normalize(result)
print(json.dumps({"objective":"T03.O1","output":normalize(result)}, sort_keys=True))

### T03.O2: Predict one-hot output dimensions and column order.
Predict the output and which rows may fit before running. Column order follows the saved basic/pro vocabulary, not the order of new records. Two categories with no drop give two coordinates per observation.

Misconception: One-hot column order is inferred anew from the order in which evaluation categories appear.

Success: Choose the encoding matrix for changed records.

Remediation: Map each category to a named coordinate.

Guided change: Compare a known category with an unseen category and arbitrary integer coding. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
encoder = OneHotEncoder(categories=[['basic','pro']], handle_unknown='ignore', drop=None, sparse_output=False).fit([['basic'],['pro']])
records = [['pro'], ['basic']] * 1
result = {'ordered_columns': encoder.get_feature_names_out(['plan']), 'encoded': encoder.transform(records), 'shape': encoder.transform(records).shape}
expected = json.loads("{\"ordered_columns\":[\"plan_basic\",\"plan_pro\"],\"encoded\":[[0,1],[1,0]],\"shape\":[2,2]}")
assert normalize(result) == expected
results["T03.O2"] = normalize(result)
print(json.dumps({"objective":"T03.O2","output":normalize(result)}, sort_keys=True))

### T03.O3: Distinguish justified ordinal encoding from arbitrary numbering.
Predict the output and which rows may fit before running. Numeric category codes alone do not establish order or equal distances. Low/medium/high may justify a stated ordinal ranking; plan names need semantic evidence, not code arithmetic.

Misconception: Consecutive integer codes prove equally spaced feature values and make any category suitable for distance calculations.

Success: Reject a false distance/order claim.

Remediation: Compare category semantics rather than numeric codes.

Guided change: Compare a known category with an unseen category and arbitrary integer coding. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
mapping = {'basic': 3, 'pro': 4, 'team': 5}
result = {'arbitrary_plan_codes': mapping, 'distance_meaning_justified': False, 'justified_ordinal_example': ['low','medium','high'], 'one_hot_alternative': [[1,0,0],[0,1,0],[0,0,1]]}
expected = json.loads("{\"arbitrary_plan_codes\":{\"basic\":3,\"pro\":4,\"team\":5},\"distance_meaning_justified\":false,\"justified_ordinal_example\":[\"low\",\"medium\",\"high\"],\"one_hot_alternative\":[[1,0,0],[0,1,0],[0,0,1]]}")
assert normalize(result) == expected
results["T03.O3"] = normalize(result)
print(json.dumps({"objective":"T03.O3","output":normalize(result)}, sort_keys=True))

### T03.O4: Recognize unknown/drop-reference ambiguity without asserting identical meaning.
Predict the output and which rows may fit before running. An all-zero unknown code does not mean a known reference category. With drop=first it can numerically coincide with the dropped basic category; preserve unknown provenance and interpret cautiously.

Misconception: If two encoded rows are numerically equal, the underlying categories necessarily have identical meaning.

Success: Choose a cautious interpretation of an all-zero row.

Remediation: Review the encoder's fixed configuration and provenance.

Guided change: Compare a known category with an unseen category and arbitrary integer coding. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
encoder = OneHotEncoder(categories=[['basic','pro']], handle_unknown='ignore', drop=None, sparse_output=False).fit([['basic'],['pro']])
result = {'drop': None, 'columns': encoder.get_feature_names_out(['plan']), 'basic': encoder.transform([['basic']]), 'unknown': encoder.transform([['team']]), 'same_meaning_guaranteed': False}
expected = json.loads("{\"drop\":null,\"columns\":[\"plan_basic\",\"plan_pro\"],\"basic\":[[1,0]],\"unknown\":[[0,0]],\"same_meaning_guaranteed\":false}")
assert normalize(result) == expected
results["T03.O4"] = normalize(result)
print(json.dumps({"objective":"T03.O4","output":normalize(result)}, sort_keys=True))

## T04 — Scaling is a fitted coordinate transformation
Prerequisite recall: T01–T02, N06; Statistics scaling.

Scaling learns a coordinate system, not a batch-specific cosmetic adjustment. z=(x−μ_train)/σ_train; StandardScaler population variance versus sample variance.

Terminology: mean, standard deviation, StandardScaler, fit/transform, zero variance.

Train [1,3] has mean 2 and population scale 1, giving [-1,1]. Test value 5 transforms to 3. Its presence must not change the training mean/scale.

Change the conditions: Freeze train mean/scale, then change test values and a constant-feature case.

### T04.O1 — critical decision AND reason: Use training-only scaling statistics and correct denominator.
Predict the output and which rows may fit before running. StandardScaler uses training population variance: mean2, variance1, scale1. A sample denominator gives2 instead. Freeze these coordinates before validation/test; a legal leaked fit is still invalid.

Misconception: Fit scaling on the whole dataset or use sample variance; either is valid because standardization is unsupervised.

Success: Repair a leaked or sample-denominator scaler and explain its coordinates.

Remediation: Revisit population/sample variance and saved training parameters.

Guided change: Freeze train mean/scale, then change test values and a constant-feature case. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = np.array([[1.],[3.]])
scaler = StandardScaler().fit(train)
leaked = StandardScaler().fit(np.vstack([train, [[5.]]]))
result = {'fit_ids': ['H01','H02'], 'mean': scaler.mean_, 'population_variance': scaler.var_, 'scale': scaler.scale_, 'test': scaler.transform([[5.]]), 'sample_variance_wrong_for_scaler': np.var(train, ddof=1), 'leaked_mean': leaked.mean_, 'whole_data_fit_allowed': False}
expected = json.loads("{\"fit_ids\":[\"H01\",\"H02\"],\"mean\":[2],\"population_variance\":[1],\"scale\":[1],\"test\":[[3]],\"sample_variance_wrong_for_scaler\":2,\"leaked_mean\":[3],\"whole_data_fit_allowed\":false}")
assert normalize(result) == expected
results["T04.O1"] = normalize(result)
print(json.dumps({"objective":"T04.O1","output":normalize(result)}, sort_keys=True))

### T04.O2: Predict transformed values and inverse units.
Predict the output and which rows may fit before running. Subtract the training mean and divide by training scale. The standardized coordinate is unitless; inverse_transform restores the original years using the same saved parameters.

Misconception: A standardized value keeps its original units and should be converted back using test-batch statistics.

Success: Choose a changed test coordinate under frozen μ/σ.

Remediation: Substitute into the stated transformation.

Guided change: Freeze train mean/scale, then change test values and a constant-feature case. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
scaler = StandardScaler().fit([[1.],[3.]])
coordinate = scaler.transform([[5.]])
result = {'coordinate': coordinate, 'inverse_years': scaler.inverse_transform(coordinate), 'formula': '(x - training_mean) / training_scale'}
expected = json.loads("{\"coordinate\":[[3]],\"inverse_years\":[[5]],\"formula\":\"(x - training_mean) / training_scale\"}")
assert normalize(result) == expected
results["T04.O2"] = normalize(result)
print(json.dumps({"objective":"T04.O2","output":normalize(result)}, sort_keys=True))

### T04.O3: Handle a constant feature according to pinned StandardScaler semantics.
Predict the output and which rows may fit before running. The pinned StandardScaler assigns scale1 to this constant training feature. Training values map to0; a new5 maps to1. Preserve the separate Math runtime's zero-variance guard rather than changing it.

Misconception: A constant feature always divides by zero in every library and must silently turn every future value into zero.

Success: Choose its documented scale/output instead of importing old runtime behavior.

Remediation: Compare the library fixture with the separately preserved math-runtime guard.

Guided change: Freeze train mean/scale, then change test values and a constant-feature case. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
scaler = StandardScaler().fit([[4.],[4.]])
result = {'variance': scaler.var_, 'scale': scaler.scale_, 'train_output': scaler.transform([[4.],[4.]]), 'new_output': scaler.transform([[5.]])}
expected = json.loads("{\"variance\":[0],\"scale\":[1],\"train_output\":[[0],[0]],\"new_output\":[[1]]}")
assert normalize(result) == expected
results["T04.O3"] = normalize(result)
print(json.dumps({"objective":"T04.O3","output":normalize(result)}, sort_keys=True))

### T04.O4: Explain that scaling does not itself prove better generalization.
Predict the output and which rows may fit before running. Scaling changes coordinates and may affect optimization or distance-based behavior. Nicer ranges do not establish better generalization; that requires a valid held-out evaluation protocol.

Misconception: Features centered on zero prove the model will generalize better regardless of evaluation.

Success: Reject a performance claim based on nicer-looking feature ranges.

Remediation: Separate numerical conditioning from held-out evaluation.

Guided change: Freeze train mean/scale, then change test values and a constant-feature case. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
raw = np.array([[4.], [6.]])
scaled = StandardScaler().fit_transform(raw)
result = {'raw': raw, 'scaled': scaled, 'numerical_purpose': 'coordinate conditioning', 'better_generalization_proved': False}
expected = json.loads("{\"raw\":[[4],[6]],\"scaled\":[[-1],[1]],\"numerical_purpose\":\"coordinate conditioning\",\"better_generalization_proved\":false}")
assert normalize(result) == expected
results["T04.O4"] = normalize(result)
print(json.dumps({"objective":"T04.O4","output":normalize(result)}, sort_keys=True))

## T05 — Compose preprocessing without leaking across folds
Prerequisite recall: T01–T04; P05 function composition.

A pipeline controls which data each learned transformation may inspect. Function composition with training-dependent parameters.

Terminology: ColumnTransformer, Pipeline, fitted state, numeric/categorical branch, fold.

First define a fold as one temporary train/validation partition using two small row-ID lists; no sklearn CV API is assumed. Numeric columns then go through SimpleImputer and StandardScaler; categorical columns through an explicit category imputer and OneHotEncoder. Every learned step sees only that fold's training IDs before transforming validation IDs. S03 later expands this model into repeated folds and API syntax.

Change the conditions: Start with a two-partition fold primer, then compose typed branches and repeat fitting per fold.

### T05.O1 — critical decision AND reason: Choose a correct branch/order and preserve the final feature schema.
Predict the output and which rows may fit before running. Numeric missing values are imputed then scaled; categorical missing labels are imputed then encoded. Fit each typed branch on training rows, concatenate in declared branch order and preserve that feature schema.

Misconception: Any pipeline ordering is valid if it produces a numeric matrix; fitting steps outside the boundary cannot leak.

Success: Repair a swapped encoder/scaler or externally fitted step and justify the order.

Remediation: Trace raw types into each named branch.

Guided change: Start with a two-partition fold primer, then compose typed branches and repeat fitting per fold. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = pd.DataFrame({'age': [10., np.nan, 30.], 'plan': ['basic','pro',np.nan]})
validation = pd.DataFrame({'age': [1003., np.nan], 'plan': ['team','basic']})
numeric = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(categories=[['basic','pro']], handle_unknown='ignore', sparse_output=False))
preprocessor = ColumnTransformer([('numeric', numeric, ['age']), ('category', categorical, ['plan'])])
preprocessor.fit(train)
result = {'fit_ids': ['H01','H02','H03'], 'branch_order': ['numeric: impute then scale', 'category: impute then encode'], 'columns': preprocessor.get_feature_names_out(), 'validation': preprocessor.transform(validation), 'invalid_order': 'scale raw plan text; or externally fit on all rows'}
expected = json.loads("{\"fit_ids\":[\"H01\",\"H02\",\"H03\"],\"branch_order\":[\"numeric: impute then scale\",\"category: impute then encode\"],\"columns\":[\"numeric__age\",\"category__plan_basic\",\"category__plan_pro\"],\"validation\":[[120.3924208577932,0,0],[0,1,0]],\"invalid_order\":\"scale raw plan text; or externally fit on all rows\"}")
assert normalize(result) == expected
results["T05.O1"] = normalize(result)
print(json.dumps({"objective":"T05.O1","output":normalize(result)}, sort_keys=True))

### T05.O2 — critical decision AND reason: Fit preprocessing inside each training fold, never on its validation rows.
Predict the output and which rows may fit before running. A fold is a temporary train/validation partition. Every learned preprocessing stage fits only that fold's training IDs, then transforms its validation IDs. Whole-development prefit leaks even though the final test is untouched.

Misconception: Fitting once on all development rows before cross-validation is safe because only the final test must stay unseen.

Success: Reject prefit whole-data preprocessing in a fold diagram.

Remediation: Redraw fit/transform arrows per fold.

Guided change: Start with a two-partition fold primer, then compose typed branches and repeat fitting per fold. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
fold_train_ids = ['H01','H02','H03']
fold_validation_ids = ['H04','H05']
train = np.array([[10.],[np.nan],[30.]])
validation = np.array([[1003.],[np.nan]])
local = SimpleImputer(strategy='median').fit(train)
invalid_prefit = SimpleImputer(strategy='median').fit(np.vstack([train, validation]))
result = {'stage_fit_ids': fold_train_ids, 'transform_only_ids': fold_validation_ids, 'fold_statistic': local.statistics_, 'validation_output': local.transform(validation), 'invalid_prefit_statistic': invalid_prefit.statistics_, 'prefit_allowed': False}
expected = json.loads("{\"stage_fit_ids\":[\"H01\",\"H02\",\"H03\"],\"transform_only_ids\":[\"H04\",\"H05\"],\"fold_statistic\":[20],\"validation_output\":[[1003],[20]],\"invalid_prefit_statistic\":[30],\"prefit_allowed\":false}")
assert normalize(result) == expected
results["T05.O2"] = normalize(result)
print(json.dumps({"objective":"T05.O2","output":normalize(result)}, sort_keys=True))

### T05.O3: Predict transformed column names/order and output shape.
Predict the output and which rows may fit before running. ColumnTransformer concatenates named branches in declaration order. The numeric age coordinate precedes basic/pro encoding; column count and names must match the saved model-input contract.

Misconception: The same output shape permits any column order, regardless of vocabulary or branch names.

Success: Select the exact joined numeric/categorical feature table.

Remediation: Read branch order and the encoder's fitted vocabulary.

Guided change: Start with a two-partition fold primer, then compose typed branches and repeat fitting per fold. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = pd.DataFrame({'age': [10., np.nan, 30.], 'plan': ['basic','pro',np.nan]})
validation = pd.DataFrame({'age': [1003., np.nan], 'plan': ['team','basic']})
numeric = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(categories=[['basic','pro']], handle_unknown='ignore', sparse_output=False))
preprocessor = ColumnTransformer([('numeric', numeric, ['age']), ('category', categorical, ['plan'])])
preprocessor.fit(train)
result = {'columns': preprocessor.get_feature_names_out(), 'shape': preprocessor.transform(validation).shape, 'table': preprocessor.transform(validation)}
expected = json.loads("{\"columns\":[\"numeric__age\",\"category__plan_basic\",\"category__plan_pro\"],\"shape\":[2,3],\"table\":[[120.3924208577932,0,0],[0,1,0]]}")
assert normalize(result) == expected
results["T05.O3"] = normalize(result)
print(json.dumps({"objective":"T05.O3","output":normalize(result)}, sort_keys=True))

### T05.O4: Distinguish fit_transform on training data from transform on new data.
Predict the output and which rows may fit before running. fit_transform belongs on authorized training rows; transform reuses fitted attributes for new rows. A second fit on evaluation rows replaces the training rule and invalidates the boundary.

Misconception: Calling fit_transform again on validation is equivalent to transform because the pipeline class has not changed.

Success: Repair a second fit on evaluation rows.

Remediation: Track learned attributes before/after each call.

Guided change: Start with a two-partition fold primer, then compose typed branches and repeat fitting per fold. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = pd.DataFrame({'age': [10., np.nan, 30.], 'plan': ['basic','pro',np.nan]})
validation = pd.DataFrame({'age': [1003., np.nan], 'plan': ['team','basic']})
numeric = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(categories=[['basic','pro']], handle_unknown='ignore', sparse_output=False))
preprocessor = ColumnTransformer([('numeric', numeric, ['age']), ('category', categorical, ['plan'])])
preprocessor.fit(train)
before = preprocessor.named_transformers_['numeric'].named_steps['simpleimputer'].statistics_.copy()
transformed = preprocessor.transform(validation)
result = {'training_median_before': before, 'training_median_after': preprocessor.named_transformers_['numeric'].named_steps['simpleimputer'].statistics_, 'validation': transformed, 'new_fit_allowed': False}
expected = json.loads("{\"training_median_before\":[20],\"training_median_after\":[20],\"validation\":[[120.3924208577932,0,0],[0,1,0]],\"new_fit_allowed\":false}")
assert normalize(result) == expected
results["T05.O4"] = normalize(result)
print(json.dumps({"objective":"T05.O4","output":normalize(result)}, sort_keys=True))

## T06 — Outlier policies and reproducible preparation reports
Prerequisite recall: T01–T05; D06, V04.

Outlier handling and reproducibility are recorded policies, not automatic cleanup. Robustness, sampling bias and explicit transformation parameters.

Terminology: outlier, clipping, fitted artifact, seed, provenance, policy, reproducibility.

A supplied training-only cap is 50 years; a test age of 80 maps to 50 only under that recorded policy. Choosing a cap after checking which value improves final-test performance is leakage.

Change the conditions: Compare preserve/investigate/cap plans, then remove a seed-only reproducibility claim.

### T06.O1 — critical decision AND reason: Choose an outlier policy without test-guided deletion or tuning.
Predict the output and which rows may fit before running. A cap50 is a supplied training-only policy, not automatic cleaning. Test80 maps to50 only under that policy. Do not tune the cap or delete rows to improve final-test scores; consider population coverage and investigate.

Misconception: Choose whichever cap or row deletion gives the best final-test score, since outliers are errors by definition.

Success: Reject a favorable-test-score clipping policy and justify investigation/train-only choice.

Remediation: Trace policy selection data and consequences for population coverage.

Guided change: Compare preserve/investigate/cap plans, then remove a seed-only reproducibility claim. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
test_age = np.array([80.])
result = {'policy': 'supplied training-only cap', 'cap_years': 50, 'transformed': np.clip(test_age, 0, 50), 'preserve_alternative': test_age, 'test_score_may_choose_cap': False, 'coverage_warning': 'investigate valid extreme observations before removal'}
expected = json.loads("{\"policy\":\"supplied training-only cap\",\"cap_years\":50,\"transformed\":[50],\"preserve_alternative\":[80],\"test_score_may_choose_cap\":false,\"coverage_warning\":\"investigate valid extreme observations before removal\"}")
assert normalize(result) == expected
results["T06.O1"] = normalize(result)
print(json.dumps({"objective":"T06.O1","output":normalize(result)}, sort_keys=True))

### T06.O2: Record fitted parameters, column order and dataset/version identities.
Predict the output and which rows may fit before running. Record data identity/hash, environment, fitted parameters, exact fit IDs, output column order, units, policy and limitations. A preprocessing artifact is more than a seed or a plausible transformed value.

Misconception: A seed and a screenshot of transformed data are sufficient to reconstruct any preparation workflow.

Success: Choose a complete preprocessing report.

Remediation: Compare a new transform with the saved artifact.

Guided change: Compare preserve/investigate/cap plans, then remove a seed-only reproducibility claim. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
scaler = StandardScaler().fit([[1.],[3.]])
result = {'dataset': 'preparation-houses.v1', 'source_revision': 3, 'source_sha256': '751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5', 'environment': 'macos-arm64-cpython-3.13.16-reference.1', 'fit_ids': ['H01','H02'], 'columns': ['age'], 'units': ['years'], 'mean': scaler.mean_, 'scale': scaler.scale_, 'seed': 42, 'schema_policy': 'reject unreviewed changes', 'limitation': 'fictional data, not population evidence'}
expected = json.loads("{\"dataset\":\"preparation-houses.v1\",\"source_revision\":3,\"source_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"environment\":\"macos-arm64-cpython-3.13.16-reference.1\",\"fit_ids\":[\"H01\",\"H02\"],\"columns\":[\"age\"],\"units\":[\"years\"],\"mean\":[2],\"scale\":[1],\"seed\":42,\"schema_policy\":\"reject unreviewed changes\",\"limitation\":\"fictional data, not population evidence\"}")
assert normalize(result) == expected
results["T06.O2"] = normalize(result)
print(json.dumps({"objective":"T06.O2","output":normalize(result)}, sort_keys=True))

### T06.O3: Separate deterministic seeds from complete reproducibility guarantees.
Predict the output and which rows may fit before running. The same seed with changed input order can select different IDs. Pin data/hash, ordering, dependency versions, split configuration and fitted schema as well as seeds.

Misconception: The same seed reproduces identical observation memberships even after the data order or dependency profile changes.

Success: Identify an unpinned ordering/library/data change.

Remediation: Use the environment/data/seed checklist.

Guided change: Compare preserve/investigate/cap plans, then remove a seed-only reproducibility claim. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
ids = ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']
reordered = list(reversed(ids)) if False else ids[1:] + ids[:1]
first, _ = train_test_split(ids, test_size=4, random_state=42)
second, _ = train_test_split(reordered, test_size=4, random_state=42)
result = {'seed': 42, 'original_train_ids': first, 'changed_order_train_ids': second, 'seed_alone_sufficient': False}
expected = json.loads("{\"seed\":42,\"original_train_ids\":[\"H06\",\"H03\",\"H02\",\"H12\",\"H05\",\"H08\",\"H04\",\"H07\"],\"changed_order_train_ids\":[\"H07\",\"H04\",\"H03\",\"H01\",\"H06\",\"H09\",\"H05\",\"H08\"],\"seed_alone_sufficient\":false}")
assert normalize(result) == expected
results["T06.O3"] = normalize(result)
print(json.dumps({"objective":"T06.O3","output":normalize(result)}, sort_keys=True))

### T06.O4: Explain limitations and abstain on unsupported schema changes.
Predict the output and which rows may fit before running. Equal width does not prove compatible features. Reject unsupported new/reordered columns explicitly and preserve the saved schema; do not silently guess a reordering or category equivalence.

Misconception: Any same-width feature matrix can be passed to the trained model without checking names or order.

Success: Choose an explicit incompatibility error rather than silently reordering features.

Remediation: Return to D06 schema audit and T05 output contract.

Guided change: Compare preserve/investigate/cap plans, then remove a seed-only reproducibility claim. Preserve the authorized fit boundary.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
expected_columns = ['age','plan_basic','plan_pro']
received_columns = ['plan_pro', 'age', 'plan_basic']
result = {'expected': expected_columns, 'received': received_columns, 'compatible': received_columns == expected_columns, 'action': 'reject incompatible schema; review explicit mapping before transform'}
expected = json.loads("{\"expected\":[\"age\",\"plan_basic\",\"plan_pro\"],\"received\":[\"plan_pro\",\"age\",\"plan_basic\"],\"compatible\":false,\"action\":\"reject incompatible schema; review explicit mapping before transform\"}")
assert normalize(result) == expected
results["T06.O4"] = normalize(result)
print(json.dumps({"objective":"T06.O4","output":normalize(result)}, sort_keys=True))

## Explicit error and policy contrasts
Predict the temporal boundary violation, constant/mode trade-offs, unknown-category error and invalid typed pipeline before running. An error is not permission to fit on evaluation rows.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train_times = [1, 2, 8]
validation_times = [3, 4]
result = {'train_times': train_times, 'validation_times': validation_times, 'future_training_information': max(train_times) >= min(validation_times), 'valid_future_forecast_boundary': False}
assert normalize(result) == json.loads("{\"train_times\":[1,2,8],\"validation_times\":[3,4],\"future_training_information\":true,\"valid_future_forecast_boundary\":false}")
print(json.dumps(normalize(result),sort_keys=True))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
train = np.array([[0.],[np.nan],[20.]])
constant = SimpleImputer(strategy='constant', fill_value=-1).fit(train)
mode = SimpleImputer(strategy='most_frequent').fit(train)
result = {'constant_policy': constant.transform(train), 'training_mode_policy': mode.transform(train), 'observed_zero_is_missing': False, 'policies_are_assumptions': True}
assert normalize(result) == json.loads("{\"constant_policy\":[[0],[-1],[20]],\"training_mode_policy\":[[0],[0],[20]],\"observed_zero_is_missing\":false,\"policies_are_assumptions\":true}")
print(json.dumps(normalize(result),sort_keys=True))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
encoder = OneHotEncoder(categories=[['basic','pro']], handle_unknown='error', sparse_output=False).fit([['basic'],['pro']])
try:
    result = encoder.transform([['team']])
except ValueError:
    result = {'tag': 'error', 'class': 'ValueError', 'action': 'reject unknown; do not refit evaluation data'}
assert normalize(result) == json.loads("{\"tag\":\"error\",\"class\":\"ValueError\",\"action\":\"reject unknown; do not refit evaluation data\"}")
print(json.dumps(normalize(result),sort_keys=True))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
try:
    result = StandardScaler().fit_transform(pd.DataFrame({'plan': ['basic','pro']}))
except ValueError:
    result = {'tag': 'error', 'class': 'ValueError', 'repair': 'categorical imputation then encoding; numeric branch imputation then scaling'}
assert normalize(result) == json.loads("{\"tag\":\"error\",\"class\":\"ValueError\",\"repair\":\"categorical imputation then encoding; numeric branch imputation then scaling\"}")
print(json.dumps(normalize(result),sort_keys=True))

## Reusable notebook template: full housing preparation report
Question → schema/availability audit → fixed splits → fold-local fit → validation transform → fixed policy → development refit → final-test transform → provenance/limitations. This notebook prepares features only; no model fitting, model comparison or cross-validation API curriculum. Validation is H09–H12 within development H01–H12; final test remains H13–H16. Explain why fitting before partitioning would invalidate the validation boundary.

In [ ]:
houses = pd.DataFrame(json.loads("{\"id\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\",\"H13\",\"H14\",\"H15\",\"H16\"],\"area\":[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16],\"age\":[0,5,null,10,0,5,null,10,0,5,null,10,0,5,null,10],\"neighborhood\":[\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\"],\"price\":[100,120,140,160,180,200,220,240,260,280,300,320,340,360,380,400],\"future_price\":[110,130,150,170,190,210,230,250,270,290,310,330,350,370,390,410],\"listed_on\":[\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\"]}"))
assert len(houses) == 16 and houses["id"].is_unique
assert hashlib.sha256(json.dumps(json.loads("{\"id\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\",\"H13\",\"H14\",\"H15\",\"H16\"],\"area\":[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16],\"age\":[0,5,null,10,0,5,null,10,0,5,null,10,0,5,null,10],\"neighborhood\":[\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\"],\"price\":[100,120,140,160,180,200,220,240,260,280,300,320,340,360,380,400],\"future_price\":[110,130,150,170,190,210,230,250,270,290,310,330,350,370,390,410],\"listed_on\":[\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\"]}"), sort_keys=True).encode()).hexdigest() == "751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5"
features = ["area","age","neighborhood"]
train = houses.iloc[:8]
validation = houses.iloc[8:12]
test = houses.iloc[12:]
def recipe():
    numeric = make_pipeline(SimpleImputer(strategy="median"),StandardScaler())
    category = make_pipeline(SimpleImputer(strategy="most_frequent"),OneHotEncoder(categories=[["A","B"]],handle_unknown="ignore",sparse_output=False))
    return ColumnTransformer([("numeric",numeric,["area","age"]),("category",category,["neighborhood"])])
fold = recipe().fit(train[features])
fold_statistics = fold.named_transformers_["numeric"].named_steps["simpleimputer"].statistics_.copy()
validation_output = fold.transform(validation[features])
np.testing.assert_array_equal(fold_statistics, [4.5,5.])
assert not set(train["id"]) & set(validation["id"])
assert not set(houses.iloc[:12]["id"]) & set(test["id"])
chosen = recipe().fit(houses.iloc[:12][features])
final_output = chosen.transform(test[features])
assert final_output.shape == (4,4) and np.isfinite(final_output).all()
assert chosen.get_feature_names_out().tolist() == ["numeric__area","numeric__age","category__neighborhood_A","category__neighborhood_B"]
report = {"dataset":"preparation-houses.v1","sha256":"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5","fold_fit_ids":train["id"].tolist(),"validation_ids":validation["id"].tolist(),"final_fit_ids":houses.iloc[:12]["id"].tolist(),"test_transform_ids":test["id"].tolist(),"feature_order":chosen.get_feature_names_out().tolist(),"fold_statistics":normalize(fold_statistics),"validation":normalize(validation_output),"final_test":normalize(final_output),"limitations":"fixed fictional data; selected policies not tuned on test; no model-performance claim"}
print(json.dumps(report,sort_keys=True))

## Clean-run summary
Run all cells in order in a fresh kernel twice. Compare normalized outputs, fit identities, schemas and analytic anchors, not screenshots or formatted library repr. Genuine Colab and other environments require actual separate verification. Seeds alone do not freeze data/order/versions. Resource checks below are trusted author observations, not OS sandbox guarantees.

In [ ]:
assert len(results) == 24
assert time.monotonic()-started < 300
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
assert rss < 2*1024**3
summary = {"families":24,"lessons":6,"seed":42,"versions":{"python":sys.version.split()[0],"numpy":np.__version__,"pandas":pd.__version__,"sklearn":sklearn.__version__},"objectives":results,"preparation_report":report}
payload = json.dumps(summary,sort_keys=True,allow_nan=False)
assert len(payload.encode()) < 1024**2
print(payload)